# Notebook 03b — Einfacher Random-Split (70 / 15 / 15)

**Ziel:** `data/splits/random/{train,val,test}.csv` erzeugen.

**Strategie:**
- Einfacher `train_test_split` aus scikit-learn, stratifiziert nach Genre.
- Split-Einheit ist das **Album** — kein Gruppen-Constraint nach Artist.
- Die Verteilung ist damit **exakt** 70 / 15 / 15 auf Albumebene.

**Wichtiger Unterschied zu Notebook 03 (Group Split):**
- Alben desselben Artists können auf verschiedene Splits verteilt sein.
- Das Modell kann Artist-Stil statt Genre-Stil lernen (Datenleck möglich).
- Dieser Ansatz dient dem **direkten Vergleich** mit dem methodisch korrekten Group Split.

**Output:** `data/splits/random/` — separater Ordner, damit `data/splits/` (Group Split) erhalten bleibt.

In [1]:
import sys
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

ROOT        = Path.cwd().parent
COVERS_DIR  = ROOT / "data" / "covers"
SPLITS_DIR  = ROOT / "data" / "splits" / "random"
SPLITS_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42

print(f"ROOT:       {ROOT}")
print(f"COVERS_DIR: {COVERS_DIR}")
print(f"SPLITS_DIR: {SPLITS_DIR}")

ROOT:       /Users/tjarek/uni/album-genre-classifier
COVERS_DIR: /Users/tjarek/uni/album-genre-classifier/data/covers
SPLITS_DIR: /Users/tjarek/uni/album-genre-classifier/data/splits/random


## 1. Disk-Scan — Cover als Ground Truth

In [2]:
rows = []
for jpg in sorted(COVERS_DIR.rglob("*.jpg")):
    album_id = jpg.stem.rsplit("_", 1)[-1]
    rows.append({
        "album_id":   album_id,
        "genre":      jpg.parent.name,
        "cover_path": str(jpg.relative_to(ROOT)),
    })

disk_df = pd.DataFrame(rows)
print(f"Cover auf Disk: {len(disk_df)}")
print(disk_df["genre"].value_counts().sort_index().to_string())

Cover auf Disk: 2549
genre
alternative_rock    253
classical           208
country             222
hiphop              235
house               274
indie_rock          263
jazz                271
metal               275
reggae              249
techno              299


## 2. Metadaten joinen (artist_id aus albums_raw.csv)

In [3]:
raw = pd.read_csv(ROOT / "data" / "albums_raw.csv")
raw = raw.drop_duplicates(subset="album_id", keep="first")

df = disk_df.merge(
    raw[["album_id", "artist_id"]],
    on="album_id",
    how="left",
)

missing_artist = df["artist_id"].isna().sum()
print(f"Alben ohne artist_id-Match: {missing_artist}")
if missing_artist > 0:
    print("WARNUNG: Diese Alben werden aus dem Split ausgeschlossen.")
    df = df.dropna(subset=["artist_id"]).reset_index(drop=True)

print(f"Alben für Split: {len(df)}")

Alben ohne artist_id-Match: 0
Alben für Split: 2549


## 3. Einfacher Random-Split 70 / 15 / 15 — stratifiziert nach Genre

Zwei Schritte:
1. 70 % Train / 30 % Temp — stratifiziert nach Genre
2. Temp → 50 % Val / 50 % Test — stratifiziert nach Genre (= je 15 % gesamt)

In [4]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["genre"],
    random_state=SEED,
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["genre"],
    random_state=SEED,
)

print("Alben pro Split:")
print(f"  train: {len(train_df)}  ({len(train_df)/len(df)*100:.1f} %)")
print(f"  val:   {len(val_df)}   ({len(val_df)/len(df)*100:.1f} %)")
print(f"  test:  {len(test_df)}  ({len(test_df)/len(df)*100:.1f} %)")

print("\nAlben pro Genre × Split:")
train_df = train_df.copy(); train_df["split"] = "train"
val_df   = val_df.copy();   val_df["split"]   = "val"
test_df  = test_df.copy();  test_df["split"]  = "test"
combined = pd.concat([train_df, val_df, test_df])
print(combined.groupby(["genre", "split"]).size().unstack(fill_value=0).to_string())

Alben pro Split:
  train: 1784  (70.0 %)
  val:   382   (15.0 %)
  test:  383  (15.0 %)

Alben pro Genre × Split:
split             test  train  val
genre                             
alternative_rock    38    177   38
classical           31    146   31
country             33    155   34
hiphop              35    165   35
house               41    192   41
indie_rock          40    184   39
jazz                41    190   40
metal               42    192   41
reggae              37    174   38
techno              45    209   45


## 4. Vergleich: Wie viele Artists sind über Splits verteilt?

Beim Random Split können Alben desselben Artists in verschiedenen Splits landen.
Diese Zelle zeigt, wie stark dieses "Datenleck" tatsächlich ist.

In [5]:
leakage = 0
for artist, sub in combined.groupby("artist_id"):
    splits_seen = sub["split"].unique()
    if len(splits_seen) > 1:
        leakage += 1

total_artists = combined["artist_id"].nunique()
print(f"Artists gesamt:                  {total_artists}")
print(f"Artists in mehreren Splits:      {leakage}  ({leakage/total_artists*100:.1f} %)")
print(f"Artists nur in einem Split:      {total_artists - leakage}  ({(total_artists-leakage)/total_artists*100:.1f} %)")
print()
print("→ Zum Vergleich Group Split (Notebook 03): 0 Artists in mehreren Splits (0.0 %)")

Artists gesamt:                  258
Artists in mehreren Splits:      229  (88.8 %)
Artists nur in einem Split:      29  (11.2 %)

→ Zum Vergleich Group Split (Notebook 03): 0 Artists in mehreren Splits (0.0 %)


## 5. CSVs schreiben

In [6]:
cols = ["album_id", "genre", "artist_id", "cover_path"]
for split, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    out_path = SPLITS_DIR / f"{split}.csv"
    split_df[cols].to_csv(out_path, index=False)
    print(f"  {split}.csv  →  {len(split_df)} Alben  →  {out_path}")

print("\nFertig. Verwende data/splits/random/ in den Training-Notebooks.")

  train.csv  →  1784 Alben  →  /Users/tjarek/uni/album-genre-classifier/data/splits/random/train.csv
  val.csv  →  382 Alben  →  /Users/tjarek/uni/album-genre-classifier/data/splits/random/val.csv
  test.csv  →  383 Alben  →  /Users/tjarek/uni/album-genre-classifier/data/splits/random/test.csv

Fertig. Verwende data/splits/random/ in den Training-Notebooks.
